In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, current_timestamp, input_file_name, from_json,get_json_object,to_json
from pyspark.sql.types import ArrayType, StringType, MapType
import os


In [ ]:
# Run once per kernel if needed:
# %pip install pyspark-ai langchain-community ollama

In [3]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.master("local[*]")
.getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/25 20:11:52 WARN Utils: Your hostname, Hariprasads-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.11.119.102 instead (on interface en0)
26/04/25 20:11:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/25 20:11:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [21]:
raw_data = "../../raw_data/folder_1/"

raw_df = (
    spark.read
    .format("binaryFile")
    .load(raw_data)
    .select( col("content").cast("string").alias("raw_json"))
    .withColumn("input_file_name", input_file_name())
)

result_df = raw_df.select(
    "raw_json",
    get_json_object("raw_json", "$.resourceType").alias("bundle_resource_type"),
    get_json_object("raw_json", "$.type").alias("bundle_type"),
    get_json_object("raw_json", "$.entry").alias("entry_raw_json"),
    "input_file_name"
)

intermediate_df_1 = result_df.select(
    "bundle_resource_type",
    "bundle_type",
    from_json("entry_raw_json", ArrayType(MapType(StringType(), StringType()))).alias("entry_array"),
    "input_file_name"
).withColumn("entries", explode("entry_array")).select(
    "bundle_resource_type",
    "bundle_type",
    col("entries"),
    "input_file_name"
)

final_df = (intermediate_df_1.select(
    "bundle_resource_type",
    "bundle_type",
    col("entries.fullUrl").alias("fullUrl"),
    col("entries.resource").alias("resource_raw_json"),
    col("entries.request").alias("request_raw_json"),
    "input_file_name"
).withColumn("resource_type", get_json_object(col("request_raw_json"), "$.url")) # keep nested JSON as raw strings so schema inference can see real objects/arrays.
    .withColumn("ingestion_timestamp", current_timestamp())
)

In [ ]:
sample_data = final_df.filter(col("resource_type") == "Claim").select(col("resource_raw_json").alias("resource")).sample(fraction=0.1, seed=42).limit(20)

In [ ]:
final_df.filter(col("resource_type") == "Claim").select(col("resource_raw_json").alias("resource")).limit(1).show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [16]:
import json
import os
from collections import Counter
import httpx
from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    DoubleType,
    LongType,
    NullType,
    StringType,
    StructField,
    StructType,
    TimestampType,
    DataType,
    )

OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:e4b")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

try:
    from pyspark_ai import SparkAI
    HAS_PYSPARK_AI = True
except Exception:
    SparkAI = None
    HAS_PYSPARK_AI = False

try:
    from langchain_community.llms import Ollama as LangchainOllama
    HAS_LANGCHAIN_OLLAMA = True
except Exception:
    LangchainOllama = None
    HAS_LANGCHAIN_OLLAMA = False

def init_spark_ai_with_ollama():
    if not HAS_PYSPARK_AI:
        return None, "pyspark-ai is not installed."
    if not HAS_LANGCHAIN_OLLAMA:
        return None, "langchain-community is not installed (needed for Ollama LLM wrapper)."

    try:
        llm = LangchainOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL)
        spark_ai = SparkAI(llm=llm)
        return spark_ai, "SparkAI initialized with local Ollama successfully."
    except Exception as exc:
        return None, f"Failed to initialize SparkAI with Ollama: {exc}"

def _sanitize_json_output(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
    if text.endswith("```"):
        text = text.rsplit("\n", 1)[0]
    return text.strip()

def _parse_llm_schema_json(text: str):
    cleaned = _sanitize_json_output(text)
    try:
        return json.loads(cleaned), cleaned
    except json.JSONDecodeError:
        start = cleaned.find("{")
        end = cleaned.rfind("}")
        if start != -1 and end != -1 and end > start:
            sliced = cleaned[start : end + 1]
            return json.loads(sliced), sliced
        raise

def _json_schema_to_spark_type(schema_obj) -> DataType:
    if not isinstance(schema_obj, dict):
        return StringType()

    schema_type = schema_obj.get("type")

    if isinstance(schema_type, list):
        schema_type = next((t for t in schema_type if t != "null"), "string")

    if schema_type == "object" or "properties" in schema_obj:
        props = schema_obj.get("properties", {})
        fields = [
            StructField(k, _json_schema_to_spark_type(v), True)
            for k, v in sorted(props.items())
        ]
        return StructType(fields)

    if schema_type == "array":
        return ArrayType(_json_schema_to_spark_type(schema_obj.get("items", {})), True)

    if schema_type == "integer":
        return LongType()
    if schema_type == "number":
        return DoubleType()
    if schema_type == "boolean":
        return BooleanType()
    if schema_type == "string":
        if schema_obj.get("format") == "date-time":
            return TimestampType()
        return StringType()

    return StringType()

def _schema_json_prompt(samples):
    sample_json_lines = [json.dumps(s, ensure_ascii=True) for s in samples[:25]]
    joined_samples = "\n".join(sample_json_lines)

    return f"""You are a PySpark schema generator.
Return ONLY JSON.
Preferred output format: a JSON object accepted by StructType.fromJson(...),
for example: {{"type": "struct", "fields": [ ... ]}}.
Alternative accepted format: JSON Schema object with type/properties.
Do not return markdown fences.

Input JSON samples:
{joined_samples}
"""

def _infer_schema_with_ollama_json(samples):
    prompt = _schema_json_prompt(samples)
    payload = {
        "model": OLLAMA_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "format": "json",
    }

    with httpx.Client(timeout=120.0) as client:
        response = client.post(f"{OLLAMA_BASE_URL}/api/chat", json=payload)
        response.raise_for_status()
        body = response.json()

    schema_json, raw_content = _parse_llm_schema_json(body["message"]["content"])


    # Native Spark StructType JSON shape.
    if isinstance(schema_json, dict) and schema_json.get("type") == "struct" and "fields" in schema_json:
        return StructType.fromJson(schema_json), raw_content

    # Wrapped response cases.
    if isinstance(schema_json, dict) and isinstance(schema_json.get("schema"), dict):
        wrapped = schema_json["schema"]
        if wrapped.get("type") == "struct" and "fields" in wrapped:
            return StructType.fromJson(wrapped), raw_content
        return _json_schema_to_spark_type(wrapped), raw_content

    # JSON Schema-like output.
    return _json_schema_to_spark_type(schema_json), raw_content

def _merge_types(left: DataType, right: DataType) -> DataType:
    if isinstance(left, NullType):
        return right
    if isinstance(right, NullType):
        return left
    if type(left) is type(right):
        if isinstance(left, StructType):
            return _merge_struct_types(left, right)
        if isinstance(left, ArrayType):
            return ArrayType(_merge_types(left.elementType, right.elementType), True)
        return left

    numeric_types = (LongType, DoubleType)
    if isinstance(left, numeric_types) and isinstance(right, numeric_types):
        return DoubleType()

    return StringType()

def _merge_struct_types(left: StructType, right: StructType) -> StructType:
    left_map = {f.name: f.dataType for f in left.fields}
    right_map = {f.name: f.dataType for f in right.fields}
    all_keys = sorted(set(left_map) | set(right_map))
    merged_fields = []

    for key in all_keys:
        merged_type = _merge_types(left_map.get(key, NullType()), right_map.get(key, NullType()))
        merged_fields.append(StructField(key, merged_type, True))

    return StructType(merged_fields)

def _infer_from_python(value) -> DataType:
    if value is None:
        return NullType()
    if isinstance(value, bool):
        return BooleanType()
    if isinstance(value, int) and not isinstance(value, bool):
        return LongType()
    if isinstance(value, float):
        return DoubleType()

    if isinstance(value, str):
        if "T" in value and "-" in value and ":" in value:
            return TimestampType()
        return StringType()

    if isinstance(value, dict):
        fields = [StructField(k, _infer_from_python(v), True) for k, v in sorted(value.items())]
        return StructType(fields)

    if isinstance(value, list):
        if not value:
            return ArrayType(StringType(), True)
        elem_type = NullType()
        for item in value:
            elem_type = _merge_types(elem_type, _infer_from_python(item))
        return ArrayType(elem_type, True)

    return StringType()

def _rows_to_json_dicts(rows):
    payloads = []
    for row in rows:
        item = row[0]
        if isinstance(item, str):
            try:
                payloads.append(json.loads(item))
            except Exception:
                pass
        else:
            if hasattr(item, "asDict"):
                payloads.append(item.asDict(recursive=True))
            elif isinstance(item, dict):
                payloads.append(item)
    return payloads

def infer_structtype_from_nested_json_samples(samples):
    merged = StructType([])
    for sample in samples:
        candidate = _infer_from_python(sample)
        if isinstance(candidate, StructType):
            merged = _merge_struct_types(merged, candidate)
    return merged

def structtype_to_code(schema: StructType) -> str:
    json_repr = json.dumps(schema.jsonValue())
    return f"StructType.fromJson({json_repr})"

def generate_schema_with_ai_or_fallback(samples):
    stats = Counter()
    for item in samples:
        if isinstance(item, dict):
            stats["dict"] += 1
        elif isinstance(item, list):
            stats["list"] += 1
        else:
            stats["other"] += 1

    spark_ai, spark_ai_msg = init_spark_ai_with_ollama()

    # AI-first path: ask local Ollama (gemma4:e4b) for schema JSON.
    try:
        ai_schema, raw_ai_output = _infer_schema_with_ollama_json(samples)
        return ai_schema, structtype_to_code(ai_schema), stats, spark_ai_msg, "ollama", raw_ai_output
    except Exception as exc:
        ai_error = str(exc)

    # Deterministic fallback when LLM output is invalid or unavailable.
    schema = infer_structtype_from_nested_json_samples(samples)
    return schema, structtype_to_code(schema), stats, spark_ai_msg, "fallback", ai_error

In [ ]:
# Collect samples from the Patient resource and generate a nested StructType schema.
sample_rows = (
    final_df.filter(col("resource_type") == "Claim")
    .select(col("resource_raw_json").alias("resource"))
    .sample(fraction=0.1, seed=42)
    .limit(50)
    .collect()
)

json_samples = _rows_to_json_dicts(sample_rows)
dynamic_schema, schema_code, sample_stats, spark_ai_status, schema_mode, ai_details = generate_schema_with_ai_or_fallback(json_samples)

print(f"SparkAI init status: {spark_ai_status}")
print(f"Schema generation mode: {schema_mode}")
print(f"Sample stats: {dict(sample_stats)}")
print("\nGenerated StructType object:\n")
print(dynamic_schema)
print("\nCopy/paste code form:\n")
print(schema_code)
print("\nAI detail (raw output or fallback error):\n")
print(ai_details)

SparkAI init status: SparkAI initialized with local Ollama successfully.
Schema generation mode: fallback
Sample stats: {'dict': 50}

Generated StructType object:

StructType([StructField('billablePeriod', StructType([StructField('end', TimestampType(), True), StructField('start', TimestampType(), True)]), True), StructField('created', TimestampType(), True), StructField('diagnosis', ArrayType(StructType([StructField('diagnosisReference', StructType([StructField('reference', StringType(), True)]), True), StructField('sequence', LongType(), True)]), True), True), StructField('facility', StructType([StructField('display', StringType(), True), StructField('reference', StringType(), True)]), True), StructField('id', StringType(), True), StructField('insurance', ArrayType(StructType([StructField('coverage', StructType([StructField('display', StringType(), True)]), True), StructField('focal', BooleanType(), True), StructField('sequence', LongType(), True)]), True), True), StructField('item',

26/04/26 06:19:36 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1054879 ms exceeds timeout 120000 ms
26/04/26 06:19:36 WARN SparkContext: Killing executors is not supported by current scheduler.
26/04/26 06:37:14 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:669)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1296)
	at 

In [ ]:
StructType([
    StructField('billablePeriod', TimestampType(), True),
    StructField('created', TimestampType(), True),
    StructField('diagnosis', StringType(), True),
    StructField('facility', StringType(), True),
    StructField('id', StringType(), True),
    StructField('insurance', StringType(), True),
    StructField('item', StringType(), True), 
    StructField('patient', StringType(), True), 
    StructField('prescription', StringType(), True), 
    StructField('priority', StringType(), True), 
    StructField('procedure', StringType(), True), 
    StructField('provider', StringType(), True), 
    StructField('resourceType', StringType(), True), 
    StructField('status', StringType(), True), 
    StructField('supportingInfo', StringType(), True), 
    StructField('total', StringType(), True), 
    StructField('type', StringType(), True), 
    StructField('use', StringType(), True)
    ])